In [1]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
# from datetime import date
import asyncio
import pandas as pd

In [3]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

In [4]:
prices_to_use = "TRADES"

# ============================================================
# CHOICES:
#    "TRADES"
#    "MIDPOINT"
#    "BID"
#    "ASK"
#    "BID_ASK"
#    "ADJUSTED_LAST"
#    "HISTORICAL_VOLATILITY"
#    "OPTION_IMPLIED_VOLATILITY"
#    "FEE_RATE"
#    "REBATE_RATE"
#    "SCHEDULE"      
# ============================================================

In [5]:
# ============================================================
# CHOOSE BETWEEN VARIABLE GROUPS     
# ============================================================

'''
lookback_period = "5 Y"
length_of_each_period = "1 day"
use_regular_trading_hours = True
'''

#'''
lookback_period = "1 M"
length_of_each_period = "1 day"
use_regular_trading_hours = True
#'''

In [6]:
# ============================================================
# INPUT SYMBOLS OR CONIDs AS A LIST
# ============================================================

symbols = ['CWB', 'ICVT']
# conIds = [320106059, 641561653]

In [ ]:
async def get_historical_closes_df(contract_list, 
                                   lookback_period, 
                                   length_of_each_period='1 day',
                                   prices_to_use='TRADES',
                                   use_regular_trading_hours=True):

    df_list = []

    for contract in contract_list:

        sym = contract.symbol

        bars = await ibkr.ib.reqHistoricalDataAsync(
            contract=contract,
            endDateTime="",          # "" means now
            durationStr=lookback_period,
            barSizeSetting=length_of_each_period,
            whatToShow=prices_to_use,
            useRTH=use_regular_trading_hours,
            formatDate=1
        )

        df = pd.DataFrame([(bar.date, bar.close) for bar in bars], columns=["date", "close"])
        df['date'] = pd.to_datetime(df['date']).dt.date
        df[sym] = df['close']
        df = df.set_index("date")
        
        df_list.append(df[sym])
        
    big_df = pd.concat(df_list, axis=1)

    today = pd.Timestamp.today().date()

    if big_df.index[-1] == today:
        big_df = big_df.iloc[:-1]

    return big_df


In [8]:
async def get_historical_prices_df(contract, 
                                   lookback_period, 
                                   length_of_each_period='1 day',
                                   prices_to_use='TRADES',
                                   use_regular_trading_hours=True):

    bars = await ibkr.ib.reqHistoricalDataAsync(
        contract=contract,
        endDateTime="",          # "" means now
        durationStr=lookback_period,
        barSizeSetting=length_of_each_period,
        whatToShow=prices_to_use,
        useRTH=use_regular_trading_hours,
        formatDate=1
    )

    
    df = pd.DataFrame(
        [
            {
                "date": bar.date,
                "open": bar.open,
                "high": bar.high,
                "low": bar.low,
                "close": bar.close,
                "volume": bar.volume,
            }
            for bar in bars
        ]
    )
    
    df['date'] = pd.to_datetime(df['date']).dt.date

    return df



In [9]:

async def main():

    await start_ibkr()

    contract_list = []
    
    for sym in symbols:
    #for conId in conIds:

        contract = Stock(sym, 'SMART', 'USD')
        await ibkr.ib.qualifyContractsAsync(contract)

        contract_list.append(contract)

        # contract = await ibkr.contract_by_conId(conId)
        # contract = Future(symbol='BRR', lastTradeDateOrContractMonth='202606', exchange='CME', currency='USD')

    df = await get_historical_closes_df(contract_list, lookback_period)
    # df = await get_historical_prices_df(contract_list[0], lookback_period)

    print(df)

In [10]:
# ============================================================
# MAIN
# ============================================================ 

await main()

#if __name__ == "__main__":
 #   asyncio.run(main())

IBKR connected: True
               CWB    ICVT
date                      
2026-07-21  103.69  116.72
2026-07-22  104.00  116.51
2026-07-23  103.17  115.90
2026-07-24  101.48  113.75
2026-07-27  101.34  113.44
2026-07-28  100.31  111.86
2026-07-29   98.39  109.68
2026-07-30  101.49  113.38
2026-07-31  101.85  113.23
2026-08-03  103.30  115.11
2026-08-04  105.35  117.12
2026-08-05  104.34  116.09
2026-08-06  102.88  114.55
2026-08-07  103.86  115.71
2026-08-10  103.61  115.14
2026-08-11  103.67  115.41
2026-08-12  105.90  118.30
2026-08-13  106.40  118.53
2026-08-14  106.60  119.02
2026-08-17  106.50  119.12
2026-08-18  104.45  116.42
2026-08-19  103.73  115.84
